## SYSTÈME DE PRÉDICTION ENVIRONNEMENTALE POUR ÉLEVAGE AVICOLE

**Problématique** : Les conditions environnementales dans les poulaillers (température, humidité, qualité de l'air) 
impactent directement la santé aviaire, la croissance et le bien-être animal.

**Objectif** : Développer un modèle IA fiable pour anticiper les conditions sur 6 heures, permettant :
- La prévention du stress thermique
- L'optimisation de la ventilation 
- La détection précoce de problèmes

**Variables cibles critiques** :
-  **Température** (°C) - Confort thermique des volailles
-  **Humidité** (%) - Respiration et hydrométrie  
-  **CO** (ppm) - Sécurité respiratoire
-  **LPG** (ppm) - Détection de fuites gaz

**Dataset** : Environmental Sensor Data (132K points) - Capteurs IoT

In [1]:
!pip install tensorflow shap

   ---------------------------------------- 0.0/547.8 kB ? eta -:--:--
   ---------------------------------------- 0.0/547.8 kB ? eta -:--:--
   ------------------- -------------------- 262.1/547.8 kB ? eta -:--:--
   ---------------------------------------- 547.8/547.8 kB 1.3 MB/s eta 0:00:00
   ---------------------------------------- 0.0/2.7 MB ? eta -:--:--
   ------- -------------------------------- 0.5/2.7 MB 2.5 MB/s eta 0:00:01
   --------------- ------------------------ 1.0/2.7 MB 2.9 MB/s eta 0:00:01
   ------------------- -------------------- 1.3/2.7 MB 2.5 MB/s eta 0:00:01
   ------------------------------ --------- 2.1/2.7 MB 2.7 MB/s eta 0:00:01
   -------------------------------------- - 2.6/2.7 MB 2.5 MB/s eta 0:00:01
   ---------------------------------------- 2.7/2.7 MB 2.5 MB/s eta 0:00:00
   ---------------------------------------- 0.0/38.1 MB ? eta -:--:--
    --------------------------------------- 0.5/38.1 MB 2.8 MB/s eta 0:00:14
   - ----------------------------


[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


### 1. IMPORTS ET CONFIGURATION

In [4]:
import os
import json
import time
import numpy as np
import pandas as pd
from pathlib import Path
from typing import Tuple, Dict, List
from datetime import datetime, timedelta

# Scikit-learn avec validation temporelle
from sklearn.preprocessing import MinMaxScaler, RobustScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit

# Visualisation avancée
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px

# TensorFlow avec optimisation
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Interprétabilité
import shap

# Configuration
import warnings
warnings.filterwarnings('ignore')

# Style des graphiques
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("✅ Libraries imported successfully")
print(f"TensorFlow: {tf.__version__}")
print(f"Keras: {keras.__version__}")

# 2. CONFIGURATION ET HYPERPARAMÈTRES
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

DATA_PATH = './Version_4_iot_telemetry_data/Version_4_iot_telemetry_data.csv'

LOOK_BACK = 24            # 24 heures historiques
FORECAST_HORIZON = 6      # Prévision sur 6 heures
TARGET_COLS = ['temp', 'humidity', 'co', 'lpg']

# Configuration d'entraînement
BATCH_SIZE = 64           # Augmenté pour stabilité
EPOCHS = 100
PATIENCE_ES = 12          # Augmenté pour patience
PATIENCE_RLR = 6
LR_FACTOR = 0.5

# Configuration validation temporelle
N_SPLITS = 5              # Pour TimeSeriesSplit
ROLLING_WINDOW_SIZE = 10000  # Pour validation glissante

# Répertoires
OUT_DIR = Path('./Version_4_artifacts')
OUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR = OUT_DIR / 'models'
SCALERS_DIR = OUT_DIR / 'scalers'
RESULTS_DIR = OUT_DIR / 'results'
MODEL_DIR.mkdir(exist_ok=True)
SCALERS_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)

print("Configuration initialisée")

✅ Libraries imported successfully
TensorFlow: 2.20.0
Keras: 3.11.2
Configuration initialisée


### 2. CHARGEMENT ET AUDIT DES DONNÉES

#### 2.1) Chargement avec vérification 

In [9]:
try:
    df_raw = pd.read_csv(DATA_PATH)
    print(f" Dataset chargé: {df_raw.shape[0]:,} lignes, {df_raw.shape[1]} colonnes")
except Exception as e:
    print(f" Erreur chargement: {e}")
    # Simulation de données pour test
    print(" Création de données de test...")
    dates = pd.date_range('2020-01-01', periods=100000, freq='1T')
    df_raw = pd.DataFrame({
        'ts': dates,
        'temp': np.random.normal(25, 5, 100000),
        'humidity': np.random.normal(60, 15, 100000),
        'co': np.random.exponential(0.005, 100000),
        'lpg': np.random.exponential(0.005, 100000),
        'device': 'simulated_device'
    })

# Audit des données
print("\n AUDIT DES DONNÉES:")
print("=" * 50)
print("Types de données:")
print(df_raw.dtypes)
print("\nStatistiques descriptives:")
print(df_raw[TARGET_COLS].describe())

# Vérification valeurs manquantes
missing_data = df_raw.isnull().sum()
print(f"\n Valeurs manquantes: {missing_data.sum()} total")
print(missing_data[missing_data > 0])

 Dataset chargé: 405,184 lignes, 9 colonnes

 AUDIT DES DONNÉES:
Types de données:
ts          float64
device       object
co          float64
humidity    float64
light          bool
lpg         float64
motion         bool
smoke       float64
temp        float64
dtype: object

Statistiques descriptives:
                temp       humidity             co            lpg
count  405184.000000  405184.000000  405184.000000  405184.000000
mean       22.453987      60.511694       0.004639       0.007237
std         2.698347      11.366489       0.001250       0.001444
min         0.000000       1.100000       0.001171       0.002693
25%        19.900000      51.000000       0.003919       0.006456
50%        22.200000      54.900000       0.004812       0.007489
75%        23.600000      74.300003       0.005409       0.008150
max        30.600000      99.900002       0.014420       0.016567

 Valeurs manquantes: 0 total
Series([], dtype: int64)


#### 2.2) Analyse de la distribution temporelle